In [2]:
import pandas as pd
from io import StringIO

# read text file and only keep trip records
with open("trips.txt", "r", encoding="cp1252", errors="replace") as f:
    t_lines = [line for line in f if line.startswith("T;")]

# parse the trip records as pandas df
df_trips = pd.read_csv(StringIO("".join(t_lines)), sep=";", header=None, engine="python")

# select only columns with useful information
df_trips = df_trips.iloc[:, [0,1,2,5,6,7,8,-3]]

df_trips.columns = [
    "record_type",
    "line_number",
    "trip_number",
    # "trip_direction",
    # "variant_code",
    "from_stop",
    "start_time_min",
    "end_time_min",
    "to_stop",
    "distance_km"
]

df_trips["distance_km"] = pd.to_numeric(df_trips["distance_km"], errors="coerce")
df_trips.head()

,record_type,line_number,trip_number,from_stop,start_time_min,end_time_min,to_stop,distance_km
0,T,u001,1001,utrvec,350,389,utrijs,12.983
1,T,u001,1003,utrvec,364,404,utrijs,12.983
2,T,u001,1005,utrvec,377,419,utrijs,12.983
3,T,u001,1007,utrvec,389,434,utrijs,12.983
4,T,u001,1009,utrvec,403,449,utrijs,12.983


In [3]:
# read text file and only keep deadhead records

with open("dhd.txt", "r", encoding="cp1252", errors="replace") as f:
    t_lines = [line for line in f if line.startswith("D;")]

# parse the deadhead records as pandas df
df_deadheads = pd.read_csv(StringIO("".join(t_lines)), sep=";", header=None, engine="python")

# select only columns with useful information
df_deadheads = df_deadheads.iloc[:, [0,1,2,3,4,5,6]]

df_deadheads.columns = [
    "record_type",
    "from_stop-to_stop",
    "time_ver_0",
    "time_ver_1",
    "time_ver_2",
    "time_ver_3",
    "distance_km",
]
df_deadheads["distance_km"] = pd.to_numeric(df_deadheads["distance_km"], errors="coerce")
df_deadheads[["from_stop", "to_stop"]] = df_deadheads["from_stop-to_stop"].str.split("-", n=1, expand=True)
df_deadheads.head()

,record_type,from_stop-to_stop,time_ver_0,time_ver_1,time_ver_2,time_ver_3,distance_km,from_stop,to_stop
0,D,amdjwp-nwggam,26,26,26,26,20.0,amdjwp,nwggam
1,D,amdpri-nwggam,27,27,27,27,20.9,amdpri,nwggam
2,D,amdpri-sllgar,25,34,25,25,19.0,amdpri,sllgar
3,D,amdpri-utrgar,40,60,40,40,28.0,amdpri,utrgar
4,D,amdpri-wbdgar,55,65,55,55,43.0,amdpri,wbdgar


In [4]:
with open("parameters.txt", "r", encoding="cp1252", errors="replace") as f:
    t_lines = [line for line in f if line.startswith("U;")]

# read text file and only keep parameters records
df_params = pd.read_csv(StringIO("".join(t_lines)), sep=";", header=None, engine="python")

# parse the parameters records as pandas df
df_params = df_params.iloc[:, [0,2,4,9,10,12]]

# select only columns with useful information
df_params.columns = [
    "record_type",
    "cost_per_bus",
    # "cost_per_minute",
    "cost_per_km",
    "energy_compsumtion_kwh/km",
    "max_charge_rate_kwh/min",
    "battery_capacity_kwh"
]

df_params.head()

,record_type,cost_per_bus,cost_per_km,energy_compsumtion_kwh/km,max_charge_rate_kwh/min,battery_capacity_kwh
0,U,244.13,0.13,1.3,2.5,199.04


In [26]:
def build_deadhead_dict(df_deadheads):
    dh_dict = {}

    for row in df_deadheads.itertuples():
        dh_dict[(row.from_stop, row.to_stop)] = (
            row.time_ver_0,
            row.time_ver_1,
            row.time_ver_2,
            row.time_ver_3
        )

    return dh_dict

def feasible_arcs(trips, deadhead_dict, max_wait = 240):
    arcs = []
    trips_list = list(df_trips.itertuples())

    for i in trips_list:
        for j in trips_list:
            if i.trip_number == j.trip_number:
                continue
            
            if i.to_stop == j.from_stop:
                travel_time = 0
            else:
                key = (i.to_stop, j.from_stop)
                
                if key not in deadhead_dict:
                    continue

                t0, t1, t2, t3 = deadhead_dict[key]
                
                # check in which interval deadhead falls
                if  331 <= i.end_time_mine <= 419 or 541 <= i.end_time_min <= 899 or 1141 <= i.end_time_min <= 1439 or 1771 <= i.end_time_min <= 1859 or 1981 <= i.end_time_min <= 1999:
                    travel_time = t0
                elif 420 <= i.end_time_mine <= 540 or 900 <= i.end_time_min <= 1140 or 1860 <= i.end_time_min <= 1980:
                    travel_time = t1
                elif 0 <= i.end_time_mine <= 330 or 1440 <= i.end_time_min <= 1770:
                    travel_time = t2
                else:
                    travel_time = t3

            slack = j.start_time_min - (i.end_time_min + travel_time)

            if 0 <= slack <= max_wait:
                arcs.append((i.trip_number, j.trip_number, travel_time, slack))

    return arcs

deadhead_dict = build_deadhead_dict(df_deadheads)
        
feasible_arcs(df_trips, deadhead_dict)



[(1001, 1006, 0, 1),
 (1001, 1008, 0, 16),
 (1001, 1010, 0, 31),
 (1001, 1012, 0, 41),
 (1001, 1014, 0, 51),
 (1001, 1016, 0, 61),
 (1001, 1018, 0, 71),
 (1001, 1020, 0, 81),
 (1001, 1022, 0, 91),
 (1001, 1024, 0, 101),
 (1001, 1026, 0, 111),
 (1001, 1028, 0, 121),
 (1001, 1030, 0, 131),
 (1001, 1032, 0, 141),
 (1001, 1034, 0, 151),
 (1001, 1036, 0, 161),
 (1001, 1038, 0, 171),
 (1001, 1040, 0, 181),
 (1001, 1042, 0, 191),
 (1001, 1044, 0, 201),
 (1001, 1046, 0, 211),
 (1001, 1048, 0, 221),
 (1001, 1050, 0, 231),
 (1003, 1008, 0, 1),
 (1003, 1010, 0, 16),
 (1003, 1012, 0, 26),
 (1003, 1014, 0, 36),
 (1003, 1016, 0, 46),
 (1003, 1018, 0, 56),
 (1003, 1020, 0, 66),
 (1003, 1022, 0, 76),
 (1003, 1024, 0, 86),
 (1003, 1026, 0, 96),
 (1003, 1028, 0, 106),
 (1003, 1030, 0, 116),
 (1003, 1032, 0, 126),
 (1003, 1034, 0, 136),
 (1003, 1036, 0, 146),
 (1003, 1038, 0, 156),
 (1003, 1040, 0, 166),
 (1003, 1042, 0, 176),
 (1003, 1044, 0, 186),
 (1003, 1046, 0, 196),
 (1003, 1048, 0, 206),
 (1003, 1